# Training Mask2Former with Dinov2 backbone for Cell Instance Segmentation

## Preparations
### Import required libraries

In [ ]:
import os
import sys
import numpy as np
import time
import torch
from PIL import Image
import cv2
import random
import pickle
import json
import pycocotools
from pycocotools.coco import COCO
from pycocotools import mask as coco_mask_util
from imgaug import augmenters
import torchvision
from torchvision.transforms import functional as F
from typing import Tuple, Union, List, Dict, Final

Since we are reusing the Dataset class of Mask R-CNN and data prepared for training that model, we need the Torchvision functions for transforms in this Notebook. Add them to the path. These functions are under vision/references/detection folder, where vision is the main folder cloned from https://github.com/pytorch/vision. Simply copy references/detection folder and place it under the folder of this Jupyter Notebook.

Make sure the same installed version is checked out. Also, install pycocotools for evaluation (pip install pycocotools).

In [ ]:
sys.path.append('references/detection')
from references.detection.engine import convert_preds_to_coco
import references.detection.transforms as T
import references.detection.utils
sys.path.append('../utils')

## Pre-processing configurations
The following configurations are used for pre-processing the images during the training. 

In [ ]:
# percentage of the width/height of each object's bounding box to use for expanding
# the box duing the training
# this is done to ensure the box fully covers the object and prevent cropped masks
PERCENTAGE_TO_EXPAND_BBOX_BOUNDARIES = 0.1

# probability of adding random blur and salt-and-pepper or additive gaussian noise to the training images
# see the image Transforms section below
P_NOISE = 0.25

# the lower and upper bounds for random scaling the images (and annotated masks) for training augmentation
MIN_RANDOM_SCALE = 0.7
MAX_RANDOM_SCALE = 1.0

# model input image size (width, height)
# The Dinov2 backbone accepts input dimensions that are both divisible by 14, hence it resizes the (1024, 800) input images 
# (originall prepared for Mask R-CNN) to (1022, 798) 
# since the ascpect ratio of Mask R-CNN processed images will not change significantly by this, we do not modify further the training 
# image sizes
MODEL_INPUT_SIZE: Final[Tuple[int, int]] = (1022, 798) 


TRANSFORM_MEAN: Final[List[float]] = [0.485, 0.456, 0.406] 
TRANSFORM_STD: Final[List[float]] = [0.229, 0.224, 0.225]

### Transforms

#### Random scaling and cropping
Random scaling, random cropping, and random horizntal flip are considered below. Furthermore, we will add random color jitter (if color images), noise addition, hue/saturation modification, random 90 degrees rotation for training. 

In [ ]:
def to_numpy(tensor):
    """
    A function to convert a torch input to numpy array.
    Args:
        tensor (torch tensor).
    Returns:
        Converted to numpy array.
    """
    return tensor.detach().cpu().numpy() if tensor.requires_grad else tensor.cpu().numpy()

# rescale function
def rescale_sample(image: Image.Image, target: Dict[str, torch.tensor], new_width: int, new_height: int) \
-> Tuple[Image.Image, Dict[str, torch.tensor]]:
    
    """
    Rescale the image and the annotated objects in it by a given factor. This function operates 
    on PIL images to ensure the resizing of the input images is identical during training and
    inferencing (both should resize PIL images). torchvision.transforms.functional.resize may have a
    slightly different behaviour than PIL.Image.Image.resize and hence is not used 
    to resize the image in torch.tensor type. 

    Args:
        image (input image in PIL.Image.Image format): Input image sample to be scaled. 
        target (dictionary):  The annotations dictionary with a keys and values as 
            defined in torchvision tutorial for Faster and Mask R-CNN (bounding 
            boxes, labels, masks, crowded, area, etc). The target dictionary should 
            have 'boxes', 'area' and optionally 'masks' as keys.  
    new_width (integer): Resized width.
    new_height (integer): Resized height.
    returns:
        Resized image in PIL format. 
        target dictionary with resized boxes ('boxes'), masks ('masks') and areas 
        ('area').
    """
    width, height = image.size
    
    # target['boxes'] = target['boxes'].mul(factor).round()
    target['boxes'] = target['boxes'].mul(torch.as_tensor([float(new_width) / width, 
                                                           float(new_height) / height] * 2 ,dtype=torch.float32)).round()
    # target['area'] = target['area'].mul(factor ** 2).round()
    target['area'] = (target['boxes'][:, 2] - target['boxes'][:, 0]) * (target['boxes'][:, 3] - target['boxes'][:, 1])
    
    if 'masks' in target:
        # note that mask here is a binary mask and interpolation has to be nearest neighbor to keep 
        # the mask as binary
        target['masks'] = F.resize(img = target['masks'], 
                                   size = [new_height, new_width], 
                                   interpolation = torchvision.transforms.InterpolationMode.NEAREST)
    
    # keep the ones with postive areas, when shrinking images, it is possible to have 0 as the side of one of the boxes
    valid_ids = target['area'] > 0
    target['boxes'] = target['boxes'][valid_ids]
    target['labels'] = target['labels'][valid_ids]
    target['masks'] = target['masks'][valid_ids]
    target['area'] = target['area'][valid_ids]
    target['iscrowd'] = target['iscrowd'][valid_ids]
    
    # we are resizing the image, which is PIL.Image.Image using cv2 here 
    # (instead of PIL.Image.Image.resize)
    # this is slower and may not be necessary at all, but since cv2.resize() is used
    # during the inference, we try to be consistent here
    
    # use different interpolation schemes depending on factor
    if new_height > height or new_width > width:
        interpolation_scheme = cv2.INTER_CUBIC
    else:
        interpolation_scheme = cv2.INTER_AREA
    
    # convert to numpy, resize and convert back to PIL
    image = cv2.resize(np.array(image), (new_width, new_height), interpolation=interpolation_scheme)
    image = Image.fromarray(image)
    
    return image, target

# random crop function
def random_crop_sample(image: Image.Image, target: Dict[str, torch.tensor], width: int, height: int) \
-> Tuple[Image.Image, Dict[str, torch.tensor]]:
    """
    Randomly crop or expand the image and update the annotated objects passed 
    in target dictionary by a given set of sizes (width and height). The input
    image will be cropped (made smaller) if at least one of the input sizes 
    (width and height) is less than the image dimensions. When cropping, the 
    bounding boxes that lies outside the newly cropped image by more than 20%
    (i.e., 20% or more of the object's area lies outside the cropped image) will
    be removed and the part of the object left inside the cropped image will be 
    blacked out (to be ignored during training). 
    If both input sizes are larger than the image sizes, the image will be 
    randomly expanded on 4 sides by zero padding to return an image with the
    specified dimensions. 
    If only one of the input sizes is less than the image sizes, the image will
    be cropped. In this case, the image will not be expanded beyond it size. 

    If no non-background object (class ID 0 for 'bg') remain in the cropped image, the original 
    image will be returned and no cropping will be performed. 

    Args:
        image (input image in PIL.Image.Image format): Input image sample to be scaled. 
        target (dictionary):  The annotations dictionary with a keys and values as 
            defined in torchvision tutorial for Faster and Mask R-CNN (bounding 
            boxes, labels, masks, crowded, area, etc). The target dictionary should 
            have 'boxes', 'area' and optionally 'masks' as keys.  
        width (integer): Width of the cropped image in pixels. 
        height (integer): Height of the cropped image in pixels. 
    returns:
        Resized image in PIL format. 
        target dictionary with resized boxes ('boxes') and areas ('area').
    """
    
    w, h = image.size
    
    image_array = np.array(image)
    rgb_image = len(image_array.shape) > 2
    
    
    if h < height and w < width:
        # randomly enlarge the image by zero padding
        x1 = np.random.randint(0, width - w)
        y1 = np.random.randint(0, height - h)
        
        if rgb_image:
            # sizes of the expanded image
            expanded_image = np.zeros((height, width, 3), dtype = 'uint8')
        else:
            expanded_image = np.zeros((height, width), dtype = 'uint8')
            
        expanded_image[y1:y1+h, x1:x1+w] = image_array    
        image = Image.fromarray(expanded_image)

        target["boxes"][:, 0] += x1
        target["boxes"][:, 1] += y1
        target["boxes"][:, 2] += x1
        target["boxes"][:, 3] += y1
        
        if 'masks' in target:
            # convert to a list first
            target['masks'] = [target['masks'][idx] for idx in range(target['masks'].shape[0])]
            for idx, mask in enumerate(target['masks']): 
                expanded_mask = torch.zeros((height, width), dtype=torch.uint8)
                expanded_mask[y1:y1+h, x1:x1+w] = mask
                target["masks"][idx] = expanded_mask

            target["masks"] = torch.stack(target["masks"], dim = 0)
        
        return image, target
    
    # keep a copy to return in case no items left in the cropped image
    boxes_tensor_copy = target["boxes"].detach().clone() if target["boxes"].requires_grad else target["boxes"].clone()
    areas_tensor_copy = target["area"].detach().clone() if target["area"].requires_grad else target["area"].clone()

    # crop the image, note that in the following, we will not zero pad
    # the image if one of the input crop sizes is larger than the image size
    if h > height:
        yc1 = np.random.randint(0, h - height)
        yc2 = yc1 + height
    else:
        yc1 = 0
        yc2 = h
    if w > width:
        xc1 = np.random.randint(0, w - width)
        xc2 = xc1 + width
    else:
        xc1 = 0
        xc2 = w

    # sizes of cropped image
    crop_width = xc2 - xc1
    crop_height = yc2 - yc1
    
    # bbox = to_numpy(target["boxes"])
    # areas = to_numpy(target["area"])
    # labels = to_numpy(target["labels"])

    # bbox[:, 0] -= xc1
    # bbox[:, 1] -= yc1
    # bbox[:, 2] -= xc1
    # bbox[:, 3] -= yc1

    # bbox[:, 0] = np.maximum(bbox[:, 0], 0)
    # bbox[:, 1] = np.maximum(bbox[:, 1], 0)
    # bbox[:, 2] = np.minimum(bbox[:, 2], crop_width)
    # bbox[:, 3] = np.minimum(bbox[:, 3], crop_height)
    # bbox = bbox.astype(int)

    # new_areas = np.maximum(0, bbox[:, 2] - bbox[:, 0]) * np.maximum(0, bbox[:, 3] - bbox[:, 1])
    
    # idxs_to_keep = np.where(new_areas > 0.33 areas)[0]
    
    target["boxes"][:, 0] -= xc1
    target["boxes"][:, 1] -= yc1
    target["boxes"][:, 2] -= xc1
    target["boxes"][:, 3] -= yc1

    target["boxes"][:, 0] = torch.maximum(torch.tensor(0), target["boxes"][:, 0])
    target["boxes"][:, 1] = torch.maximum(torch.tensor(0), target["boxes"][:, 1])
    target["boxes"][:, 2] = torch.minimum(torch.tensor(crop_width), target["boxes"][:, 2])
    target["boxes"][:, 3] = torch.minimum(torch.tensor(crop_height), target["boxes"][:, 3])
    
    new_areas = (torch.maximum(torch.tensor(0), target["boxes"][:, 2] - target["boxes"][:, 0]) * 
                 torch.maximum(torch.tensor(0), target["boxes"][:, 3] - target["boxes"][:, 1]))

    # TODO: 0.33 of full objects (that are cut here by the crop) is fine, 
    # however, if the object was a partial object to begin with, then 0.33 should be much more
    idxs_to_keep = torch.where(new_areas > 0.33 * target["area"])[0]
    
    num_objects = len(idxs_to_keep)
   
    if num_objects > 0 and (target["labels"][idxs_to_keep] > 0).any():
        # crop the image first
        image = image.crop((xc1, yc1, xc2, yc2))
        # black out partial objects remaining (we are not keeping these objects)
        image_array = np.array(image)
        
        for i in range(target["boxes"].shape[0]):
            if i in idxs_to_keep or new_areas[i] == 0:
                continue
            # blackout the image
            # xtl, ytl, xbr, ybr = bbox[i]
            xtl, ytl, xbr, ybr = to_numpy(target["boxes"][i]).astype(int)
            if rgb_image:
                image_array[ytl:ybr, xtl:xbr, :] = (0, 0, 0)
            else:
                image_array[ytl:ybr, xtl:xbr] = 0
    
        image = Image.fromarray(image_array)
        
        # target["boxes"] = torch.as_tensor(bbox[idxs_to_keep], dtype = torch.float32)
        # target["area"] = torch.as_tensor(new_areas[idxs_to_keep], dtype = torch.float32)
        # target["labels"] = torch.as_tensor(labels[idxs_to_keep], dtype = torch.int64)
        target["boxes"] = target["boxes"][idxs_to_keep]
        target["labels"] = target["labels"][idxs_to_keep]
        target["area"] = new_areas[idxs_to_keep]
        
        if 'masks' in target:
            # target["masks"] = torch.as_tensor(target["masks"][idxs_to_keep, yc1: yc2, xc1: xc2], dtype = torch.uint8)
              target["masks"] = target["masks"][idxs_to_keep, yc1: yc2, xc1: xc2]
            
        target["iscrowd"] = torch.zeros((num_objects,), dtype=torch.int64)    
    else:
        # revert back the boxes/areas
        target["boxes"] = boxes_tensor_copy        
        target["area"] = areas_tensor_copy
        
    return image, target

In [ ]:
# reszie transform class
class ImgResize(object):
    """
    Class for resizing the image to a target size and without keeping the aspect ratio the same. 
    """
    def __init__(self, target_size: Tuple[int, int]):
        """
        Args: 
            target_size (tuple): Target size in (width, height) format. 
        """
        self.width: int = target_size[0]
        self.height: int = target_size[1]
        

    def __call__(self, image, target):
        """
        Resize the image and the annotations to the given target size. Note that the input image should 
        be in  PIL format unlike the transforms class under torchvision.references.detection.tranforms.py 
        that takes tensors. So when combined with other transforms, this should be called first on 
        the PIL image before the image is converted to Tensor. 
        
        Args:
            image (input image in PIL.Image.Image format): Input image sample to be scaled. 
            target (dictionary):  The annotations dictionary with a keys and values as 
                defined in torchvision tutorial for Faster and Mask R-CNN (bounding 
                boxes, labels, masks, crowded, area, etc). The target dictionary should 
                have 'boxes', 'area' and optionally 'masks' as keys.  
        Returns: 
            Resized image and annotations. The transform may change the aspect ratio of the input image. 
        """    
        return rescale_sample(image=image, target=target, new_width=self.width, new_height=self.height)

# random scale transform class
class RandomScale(object):
    """
    Class for random scaling the image while keeping the aspect ratio the same. 
    Scale factor is chosen randomly from a given range of scales.
    """
    def __init__(self, min_range, max_range):
        """
        Args: 
            min_range (float): The lower bound on the scale factor.
            max_range (float): The upper bound on the scale factor. 
        """
        if min_range >= max_range:
            print('[ERROR]: min range for scaling should be less than or equal to the max range.')
            print('The class was not instantiated.')
            return
        self.min_range = min_range
        self.max_range = max_range

    def __call__(self, image, target):
        """
        Scale the image and the annotations by a factor randomly (uniformly) chosen 
        from [self.min_range, self.max_range]. Note that the input image should be in 
        PIL format unlike the transforms class under torchvision.references.detection.tranforms.py 
        that takes tensors. So when combined with other transforms, this should be called first on 
        the PIL image before the image is converted to Tensor. 
        
        Args:
            image (input image in PIL.Image.Image format): Input image sample to be scaled. 
            target (dictionary):  The annotations dictionary with a keys and values as 
                defined in torchvision tutorial for Faster and Mask R-CNN (bounding 
                boxes, labels, masks, crowded, area, etc). The target dictionary should 
                have 'boxes', 'area' and optionally 'masks' as keys.  
        Returns: 
            Scaled image and annotations by a randomly chosen factor in 
                [self.min_range, self.max_range]. The return image is the same size as the input
                image and the aspect ratio is kept the same during scaling. If the 
                scale factor is less than 1, the returned image is zero padded to the
                same size as the input image. 
        """    
        w, h = image.size
        factor = random.random() * (self.max_range - self.min_range) + self.min_range
        height, width = int(h * factor), int(w * factor)
        image, target = rescale_sample(image, target, width, height)
        image, target = random_crop_sample(image, target, w, h)
        if factor > 1 and (image.size[0] != w or image.size[1] != h):
            # in case factor > 1 and the cropped sub-image does not have any object, the scaled image 
            # is returned, rescale it back
            image, target = rescale_sample(image, target, w, h)
        return image, target

#### Other Augmentations
In the following, we define classes for the following additional augmentation of the training images. None of the transforms below change the bounding boxes. They only change the image 'quality':
- Gaussian Blur with sigma randomly selected between 0, 2.0
- Drop-out (salt and pepper noise) with p randomly selected between 0, 0.02, where p is the percentage of dropped out pixels
- Additive Gaussian Noise with sigma randomly selected between 0, 10 out of 255

In [ ]:
# in the following classes, we use imgaug package to modify images
class ImgAugTransform(object):
    """
    Class to randomly modify the image by adding Gaussian blur (with sigma randomly
    selected between 0, 2.0), Dropout (with p randomly selected between 0, 0.02, where
    p is the percentage of dropped out pixels) and additive Guassian noise (with sigma
    randomly selected between 0, 10 out of 255 for all channel); 
    each modification is applied indepedently with probability 0.25 
    (Note: Either of drop out or additive Gaussian is applied at any time a modification 
    is made)
    No change is made to the annotations. 
    """
    
    def __init__(self, p=0):
        self.aug = augmenters.Sequential([
            # Gaussian blur with p probability
            augmenters.Sometimes(p, augmenters.GaussianBlur(sigma=(0, 2.0))),
            # either of Droput or AdditiveGaussianNoise with probability p
            augmenters.Sometimes(p,
                                 augmenters.OneOf(
                                      [augmenters.Dropout(p=(0, 0.02)), 
                                       augmenters.AdditiveGaussianNoise(scale=(0, 10.0))]
                                ))
            ])

    def __call__(self, image, target):
        """ 
        Args:
            image (input image in PIL.Image format): Input image sample to be modified. 
            target (dictionary):  The annotations dictionary with a keys and values as 
                defined in torchvision tutorial for Faster and Mask R-CNN (bounding 
                boxes, labels, masks, crowded, area, etc).
        Returns: 
            modified image and annotations (no change to annotations). 
        """
        # imgaug accept images in numpy H x W x 3 RGB format, 
        # convert the PIL image to a numpy array before applying the
        # transforms
        outImg = self.aug.augment_image(np.array(image))
        # convert back to PIL image before returning
        return Image.fromarray(outImg), target

In [ ]:
def get_transform(train: bool = True) -> references.detection.transforms.Compose:
    # no resizing is needed as Dinov2 automatically resize the input images
    # we should make sure the resizing does not significantly change the apsect ratio 
    trsfms = []
    if train:
        # random noise addition and random scale as defined above, 
        # we call these before PILToTensor as these classes 
        # operates on PIL images
        trsfms.append(ImgAugTransform(p=P_NOISE))
        trsfms.append(RandomScale(MIN_RANDOM_SCALE, MAX_RANDOM_SCALE))
        trsfms.append(T.PILToTensor())
        # ToTensor() has been removed from references.detection.transforms 
        # in newer torchvision versions and has been replaced by ToPILTensor
        trsfms.append(T.RandomHorizontalFlip(0.5))
    else:
        # test set (only convert to Tensor)
        trsfms.append(T.PILToTensor())

    trsfms.append(T.ConvertImageDtype(torch.float32)) # torch.int8 will be divided by 255 here before normalization
    trsfms.append(T.Normalize(mean=TRANSFORM_MEAN, std=TRANSFORM_STD))
    return T.Compose(trsfms)

## Data Model
The dataset class as well as the training and evaluation scripts are adopted from the Mask2Former fine-tuning tuturial Notebook here:  https://github.com/NielsRogge/Transformers-Tutorials/blob/master/MaskFormer/Fine-tuning/Fine_tune_MaskFormer_on_an_instance_segmentation_dataset_(ADE20k_full).ipynb. This Notebook provides the dataset class for the pre-processed annotated data (resized and cropped) as prepared for training our Mask R-CNN model (800 x 1024 crops). 

NOTE: The labels (class IDs) for the pre-processed annotated data for Mask R-CNN model training starts with 1 (0 is reserved for background), while the class IDs for Mask2Former model starts from 0 (the last ID will be used for backgound). We apply this offset in the dataset class below to use the same set of training for both models. 

### Dataset for already pre-processed annotated data
This dataset is built by passing the location of the pre-processed images and masks folders. The code below assumes the images and the corresponding masks use the same name. For each image with name `sample_name` (saved as .jpg or .png) under the images folder, there should be a mask with the same name and .png extension (`sample_name.png`) under the masks folder. For a faster training, pre-process the images and use the dataset below.

In [ ]:
class CellMaskDataset(torch.utils.data.Dataset):
    def __init__(self, images_path: str,
                 masks_path: str, 
                 annots_in_coco_rle_format: bool,
                 transforms: references.detection.transforms.Compose) -> None:
        
        self.images_path = images_path
        self.masks_path = masks_path
        self.transforms = transforms
        # load all images and masks
        # the assumption is the image and its mask annotation use the same name
        self.imgs = list(sorted(os.listdir(images_path)))
        self.masks = list(sorted(os.listdir(masks_path)))
        self.coco_rle_format = annots_in_coco_rle_format

        if len(self.imgs) != len(self.masks):
            print("[ERROR]: The list of images and masks are not consistent")
            return
        
        for i, img_filename in enumerate(self.imgs):
            # drop the image/mask filename extension 
            # (anything after the last '.' in the filename is considered as extension)
            img_name = ".".join(img_filename.strip().split('.')[:-1])
            mask_name = ".".join(self.masks[i].strip().split('.')[:-1])
            if img_name != mask_name:
                print("[ERROR]: Inconsistent mask file :{} found for image file: {}".format(mask_name, img_name))
     

    def __getitem__(self, idx: int):
        # load images and masks
        img_path = os.path.join(self.images_path, self.imgs[idx])
        mask_path = os.path.join(self.masks_path, self.masks[idx])
        # read the image, do not change the format
        # depending on the set color_depth, the values will be in [0, 2^color_depth - 1]

        # the model expect the image in RGB format, convert grayscale images to RGB
        img = Image.open(img_path).convert('RGB')
            
        # use OpenCV to read the image, then convert to RGB and create a PIL Image object
        # opencv_img = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)
        # if len(opencv_img.shape) > 2:
            # convert BGR to RGB
        #     opencv_img = cv2.cvtColor(opencv_img, cv2.COLOR_BGR2RGB)
        # convert to PIL.Image
        # img = Image.fromarray(opencv_img)
        
        image_width, image_height = img.size
        
        if self.coco_rle_format:
            
            boxes: List[List] = []
            labels: List[int] = []
            masks: List[np.ndarray] = []
            
            # load the annotations
            filehandler = open(mask_path, 'rb')
            annots = pickle.load(filehandler)
            filehandler.close()
            
            for record in annots['annotations']:
                xmin, ymin, xmax, ymax = record['bbox']
                # no need to check the validity 
                if xmin >= xmax or ymin >= ymax:
                    continue

                # class IDs start from 0 in Mask2Former
                labels.append(int(record['category_id']) - 1)
                mask: np.ndarray = coco_mask_util.decode(record['segmentation'])
                # mask in full image resolution
                full_res_mask: np.ndarray = np.zeros((image_height, image_width), np.uint8)
                full_res_mask[ymin:ymax, xmin:xmax] = mask
                
                masks.append(full_res_mask)
                
                # expand the bounding box if needed
                # bounding boxes are not really needed for training Mask2Former and we can drop the part below
                delta_x = int(PERCENTAGE_TO_EXPAND_BBOX_BOUNDARIES * (xmax - xmin) / 2)
                delta_y = int(PERCENTAGE_TO_EXPAND_BBOX_BOUNDARIES * (ymax - ymin) / 2)
            
                delta_x = max(1, delta_x)
                delta_y = max(1, delta_y)
            
                xmin = max(0, xmin - delta_x)
                ymin = max(0, ymin - delta_y)
                xmax = min(image_width, xmax + delta_x)
                ymax = min(image_height, ymax + delta_y)
                
                boxes.append([xmin, ymin, xmax, ymax])
                
            num_objs = len(boxes)
            # combine all the masks
            masks = np.array(masks)
            labels = np.array(labels)
        else:
            # This Mask encoding below is deprecated and should not be used
            
            # loaded_mask is a m x image_height x image_width array (the same size as the input image)
            # the assumption here is instances are encoded as different levels
            # with 0 being the background
            # each element in mask is an np.uint16 (unsigned 16 bits)
            # to support overlapping objects, objects with overlaps are reported in 
            # different arrays (loaded_mask[i])
            # so we simply need to extract masks from all these arrays
            loaded = np.load(mask_path)
            loaded_masks = loaded['saved_masks']
            loaded_labels = loaded['saved_labels']
        
            num_objs = 0
        
            masks = []
            labels = []
            for i in range(loaded_masks.shape[0]):
                # instances are encoded as different gray levels
                # the results are sorted
                # first id [0] is the background, so remove it
                obj_ids = np.unique(loaded_masks[i])[1:]
        
                num_objs += len(obj_ids)
        
                # split the color-encoded mask into a set
                # of binary masks
                masks.append((loaded_masks[i] == obj_ids[:, None, None]).astype(np.uint8))
                # make sure labels are mapped correctly to masks
                labels += list(loaded_labels[obj_ids - 1])
        
            # combine all the masks
            masks = np.concatenate(masks, axis=0)
            # we subtract one because Mask2Former labels start from 0
            labels = np.array(labels) - 1
        
            # get bounding box coordinates for each mask, not needed for Mask2Former training
            boxes = []
            valid_ids = []
            for i in range(num_objs):
                pos = np.where(masks[i])
                xmin = np.min(pos[1])
                xmax = np.max(pos[1])
                ymin = np.min(pos[0])
                ymax = np.max(pos[0])
            
                if xmin >= xmax or ymin >= ymax:
                    continue
            
                delta_x = int(PERCENTAGE_TO_EXPAND_BBOX_BOUNDARIES * (xmax - xmin) / 2)
                delta_y = int(PERCENTAGE_TO_EXPAND_BBOX_BOUNDARIES * (ymax - ymin) / 2)
            
                delta_x = max(1, delta_x)
                delta_y = max(1, delta_y)
            
                xmin = max(0, xmin - delta_x)
                ymin = max(0, ymin - delta_y)
                xmax = min(image_width, xmax + delta_x)
                ymax = min(image_height, ymax + delta_y)
            
                valid_ids.append(i)
                boxes.append([xmin, ymin, xmax, ymax])
        
            num_objs = len(valid_ids)
            labels = labels[valid_ids]
            masks = masks[valid_ids]
            
        # there are more than 1 class
        labels = torch.as_tensor(labels, dtype=torch.int64)    
            
        # convert everything into a torch.Tensor
        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        
        # masks, at this point the masks are binary (0, 1) so uint8 is fine
        masks = torch.as_tensor(masks, dtype=torch.uint8)

        image_id = torch.tensor([idx])
        area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0])
        
        # suppose all instances are not crowd
        iscrowd = torch.zeros((num_objs,), dtype=torch.int64)

        target = {}
        target["boxes"] = boxes
        target["labels"] = labels
        target["masks"] = masks
        target["image_id"] = image_id
        target["area"] = area
        target["iscrowd"] = iscrowd

        if self.transforms is not None:
            img, target = self.transforms(img, target)

        return img, target

    def __len__(self):
        return len(self.imgs)

#### A function to convert our dataset to COCO dataset format
This function is needed for efficient evaluation. Similar to the dataset class, the labels (class IDs) should start from 0. 

In [ ]:
from tqdm import tqdm
def convert_to_coco_api(images_path: str, masks_path: str, annots_in_coco_rle_format: bool=True):
    # load all images and masks
    # the assumption is the image and its mask annotation use the same name
    imgs = list(sorted(os.listdir(images_path)))
    masks = list(sorted(os.listdir(masks_path)))
    
    if len(imgs) != len(masks):
        print("[ERROR]: The list of images and masks are not consistent")
        return False, {}
    
    for i, img_filename in enumerate(imgs):
        # drop the image/mask filename extension 
        # (anything after the last '.' in the filename is considered as extension)
        img_name = ".".join(img_filename.strip().split('.')[:-1])
        mask_name = ".".join(masks[i].strip().split('.')[:-1])
        if img_name != mask_name:
            print("[ERROR]: Inconsistent mask file :{} found for image file: {}".format(mask_name, img_name))
            return False, {}
    # the index for annotations starts at 1
    annots_id = 1
    categories = set()
    json_annotations = {"images": [], "categories": [], "annotations": []}
    for idx in tqdm(range(len(imgs))):
        # load the image
        img_path = os.path.join(images_path, imgs[idx])
        # read the image, we only read the image to get the size of it
        # so no need to change the format (BGR to RGB) or convert to PIL 
        opencv_img = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)
        image_height, image_width = opencv_img.shape[:2]
        img_dict = {}
        img_dict["id"] = idx + 1
        img_dict["file_name"] = imgs[idx]
        img_dict["height"] = image_height
        img_dict["width"] = image_width
        json_annotations["images"].append(img_dict)

        # load the annotations (masks)
        mask_path = os.path.join(masks_path, masks[idx])
        
        if annots_in_coco_rle_format:
            filehandler = open(mask_path, 'rb')
            annots = pickle.load(filehandler)
            filehandler.close()
                
            for record in annots['annotations']:
                record["image_id"] = idx + 1
                # subtract 1 because labels start from 0 in Mask2Former
                record['category_id'] = record['category_id'] - 1
                categories.add(record['category_id'])
                record['bbox'] = [int(v) for v in record['bbox']]
                xmin, ymin, xmax, ymax = record['bbox']
                if "segmentation" in record:
                    mask: np.ndarray = coco_mask_util.decode(record['segmentation'])
                    # mask in full image resolution
                    full_res_mask: np.ndarray = np.zeros((image_height, image_width), np.uint8)
                    full_res_mask[ymin:ymax, xmin:xmax] = mask
                    record["segmentation"] = coco_mask_util.encode(np.asarray(full_res_mask, order="F"))
                    record["segmentation"]['counts'] = record["segmentation"]['counts'].decode('utf8')
                # expand the bounding box if needed
                delta_x = int(PERCENTAGE_TO_EXPAND_BBOX_BOUNDARIES * (xmax - xmin) / 2)
                delta_y = int(PERCENTAGE_TO_EXPAND_BBOX_BOUNDARIES * (ymax - ymin) / 2)
                
                delta_x = max(1, delta_x)
                delta_y = max(1, delta_y)
                
                xmin = max(0, xmin - delta_x)
                ymin = max(0, ymin - delta_y)
                xmax = min(image_width, xmax + delta_x)
                ymax = min(image_height, ymax + delta_y)
                # convert to xywh
                record['bbox'] = [xmin, ymin, xmax - xmin, ymax - ymin]
                record["area"] = (ymax - ymin) * (xmax - xmin)
                record["iscrowd"] = 0
                record["id"] = annots_id
                
                json_annotations["annotations"].append(record)
                annots_id += 1
        else:
            loaded = np.load(mask_path)
            loaded_masks = loaded['saved_masks']
            loaded_labels = loaded['saved_labels']
            
            num_objs = 0
        
            masks = []
            labels = []
            for i in range(loaded_masks.shape[0]):
                # instances are encoded as different gray levels
                # the results are sorted
                # first id [0] is the background, so remove it
                obj_ids = np.unique(loaded_masks[i])[1:]
        
                num_objs += len(obj_ids)
        
                # split the color-encoded mask into a set
                # of binary masks
                masks.append((loaded_masks[i] == obj_ids[:, None, None]).astype(np.uint8))
                # make sure labels are mapped correctly to masks
                labels += list(loaded_labels[obj_ids - 1])
        
            # combine all the masks
            masks = np.concatenate(masks, axis=0)
            # subtract 1 as labels start from 0 in Mask2Former
            labels = np.array(labels) - 1
        
            # get bounding box coordinates for each mask
            boxes = []
            valid_ids = []
            for i in range(num_objs):
                pos = np.where(masks[i])
                xmin = np.min(pos[1])
                xmax = np.max(pos[1])
                ymin = np.min(pos[0])
                ymax = np.max(pos[0])
            
                if xmin >= xmax or ymin >= ymax:
                    continue
            
                delta_x = int(PERCENTAGE_TO_EXPAND_BBOX_BOUNDARIES * (xmax - xmin) / 2)
                delta_y = int(PERCENTAGE_TO_EXPAND_BBOX_BOUNDARIES * (ymax - ymin) / 2)
            
                delta_x = max(1, delta_x)
                delta_y = max(1, delta_y)
            
                xmin = max(0, xmin - delta_x)
                ymin = max(0, ymin - delta_y)
                xmax = min(image_width, xmax + delta_x)
                ymax = min(image_height, ymax + delta_y)
            
                valid_ids.append(i)
                boxes.append([xmin, ymin, xmax, ymax])
        
            num_objs = len(valid_ids)
            labels = labels[valid_ids]
            masks = masks[valid_ids]

            for i, mask in enumerate(masks):
                record = {}
                record["image_id"] = idx + 1
                record['category_id'] = labels[i] # 1 is already subtracted
                categories.add(record['category_id'])
                record["segmentation"] = coco_mask_util.encode(np.asarray(mask, order="F"))
                record["segmentation"]['counts'] = record["segmentation"]['counts'].decode('utf8')
                xmin, ymin, xmax, ymax = [int(v) for v in boxes[i]]
                #convert to xywh
                record['bbox'] = [xmin, ymin, xmax - xmin, ymax - ymin]
                record["area"] = (ymax - ymin) * (xmax - xmin)
                record["iscrowd"] = 0
                record["id"] = annots_id
                
                json_annotations["annotations"].append(record)
                annots_id += 1 
            
    json_annotations["categories"] = [{"id": i} for i in sorted(categories)]

    return True, json_annotations

#### Dataset prepration for option 3
----------------

Define LABEL_MAP, a mapping between class IDs and class names used in the annotations, with class IDs starting from 1  (0 is reseved for background). This is not needed for the dataset, but during the training. 

In [ ]:
# make sure these folders are generated in advance
SET_NAME = 'sets_1_2_3_6_to_41_3_class'
TRAIN_IMAGE_FOLDER = os.path.join(os.getcwd(), 'data/' + SET_NAME + '/images/train')
TRAIN_MASK_FOLDER = os.path.join(os.getcwd(),'data/' + SET_NAME + '/masks/train')
TEST_IMAGE_FOLDER = os.path.join(os.getcwd(),'data/' + SET_NAME + '/images/test')
TEST_MASK_FOLDER = os.path.join(os.getcwd(),'data/' + SET_NAME + '/masks/test')

# mapping between the class IDs and class names for the annotated data 
# labels start from 0
LABEL_MAP = {0: 'cell', 1: 'bead', 2: 'cage'}
REVERESE_LABEL_MAP = {v: k for k, v in LABEL_MAP.items()}
# needed in case masks of different instances cannot be overlapping
ORDERED_CLASS_NAMES = ['cage', 'cell', 'bead']

train_dataset = CellMaskDataset(images_path=TRAIN_IMAGE_FOLDER, masks_path=TRAIN_MASK_FOLDER,
                                annots_in_coco_rle_format = True,
                                transforms = get_transform(train=True))

# test_dataset = CellMaskDataset(images_path=TEST_IMAGE_FOLDER, masks_path=TEST_MASK_FOLDER,
#                                 annots_in_coco_rle_format = True,
#                                 transforms = get_transform(train=False))

In [ ]:
import json
success, json_annots = convert_to_coco_api(images_path=TEST_IMAGE_FOLDER, masks_path=TEST_MASK_FOLDER)
with open('test_annotations.json', 'w') as file:
    json.dump(json_annots, file)

from references.detection.coco_utils import CocoDetection
# note that this is a modified version of torchvision.datasets.CocoDetection
test_dataset = CocoDetection(img_folder=TEST_IMAGE_FOLDER, ann_file='test_annotations.json', 
                             transforms=get_transform(train=False))
# os.remove('test_annotations.json')

## Model Definition

In [ ]:
from transformers import Dinov2Config, Dinov2Model, Mask2FormerConfig, Mask2FormerForUniversalSegmentation

def get_mask2former_instance_segmentation_model_with_dinov2_backbone(
    id2label: Dict[int, str], 
    model_type: str, 
    with_registers: bool
):

    # transformer layer outputs to use
    output_indices_map: Dict[str, List[int]] = {
        "small": [6, 8, 10, 12], 
        "base":  [6, 8, 10, 12], 
        "large": [18, 20, 22, 24], 
        "giant": [34, 36, 38, 40]
    }
    
    if model_type.lower() in output_indices_map.keys():
        if with_registers:
            dinov2_checkpoint_str: str = "dinov2-with-registers-" + model_type.lower()
        else:
            dinov2_checkpoint_str: str = "dinov2-" + model_type.lower()
        
        output_indices: List[int] = output_indices_map[model_type.lower()] 
    else:
        dinov2_checkpoint_str: str = "dinov2-base"
        output_indices: List[int] = output_indices_map["base"]
        print(f"[ERROR] Incorrect model type passed {model_type}! Using the base model by default.")
        
        

    # store Dinov2 weights locally to reload them again, only do it if already not loaded locally
    if not os.path.exists(os.path.join(MODEL_PATH, dinov2_checkpoint_str + ".pth")):
        dinov2_model = Dinov2Model.from_pretrained("facebook/" + dinov2_checkpoint_str, out_indices=output_indices)
        torch.save(dinov2_model.state_dict(), os.path.join(MODEL_PATH, dinov2_checkpoint_str + ".pth"))

    # create Mask2Former config for semantic segmentation with Dinov2 backbone
    
    mask2former_checkpoint = "facebook/mask2former-swin-large-coco-instance"
    
    model_config = Mask2FormerConfig.from_pretrained(mask2former_checkpoint)
    model = Mask2FormerForUniversalSegmentation.from_pretrained(mask2former_checkpoint,
                                                                id2label=id2label,
                                                                ignore_mismatched_sizes=True)
    model_config = model.config
    model_config.backbone_config = Dinov2Config.from_pretrained("facebook/" + dinov2_checkpoint_str, out_indices=output_indices)

    

    
    # instantiate Mask2Former model with Dinov2 backbone (random weights)
    model = Mask2FormerForUniversalSegmentation(model_config)

    # load Dinov2 weights into Mask2Former backbone
    dinov2_backbone = model.model.pixel_level_module.encoder
    dinov2_backbone.load_state_dict(torch.load(os.path.join(MODEL_PATH, dinov2_checkpoint_str + ".pth")))

    # freeze all the weights in Dinov2 backbone
    # for param in dinov2_backbone.parameters():
    #     param.requires_grad_(False)

    # this is for freezing the backbone in Mask2Former, it should be the same as above
    for param in model.model.pixel_level_module.encoder.parameters():
        param.requires_grad_(False)

    return model

## Training
### Training parameters

In [ ]:
TRAIN_BATCH_SIZE = 4
OPTIMIZER = 'Adam' # can be set to 'SGD' as well for stochastic Gradient Descent
DINOV2_BACKBONE_TYPE = 'base'
LEARNING_RATE = 2e-5
NUM_EPOCHS = 8
# use a postive number for Step LR, any number less than 1 means use One-Cycle LR
LR_DECAY_STEPS = -1
MODEL_PATH = 'checkpoints'

### Data loaders

It looks like Mask2Former can support overlapping instance masks (the dataset in the tutorial example had non-overlapping instance masks). I need to investigate further to be sure. But it seems, we do not need to define an order of objects when creating the instance masks for overlapping objects. So `collate_fn_2` below should be used. However, if the masks cannot be overlapping, We create the masks of overlapping objects in this order: bg, cage, cell, and then bead as cells can be inside cages (creating holes in cage masks), and beads can potentially be over the cells (creating holes). In this case, `collate_fn_1` should be used.

Note that `test_dataset` below is using a different class (`torchvision.datasets.CocoDetection` instead of `CellMaskDataset` defined above). We do not need any special collate function for the test_data_loader as we only use it for COCO evaluation.

In [ ]:
from transformers import Mask2FormerImageProcessor

# we need to convert the masks to a set of binary masks and a bunch of classes for training Mask2Fomer
# to simplify, we use the already implemented Hugging Face preprocessor for this conversion
# we pass all the other flags as False as the image is already augmented and normalized
# index 0 will be used 
hg_preprocessor = Mask2FormerImageProcessor(ignore_index=-1, reduce_labels=False, do_resize=False, do_rescale=False, do_normalize=False)

def generate_instance_segmentation(target: dict, 
                                   label_map: Dict[int, str] = LABEL_MAP, 
                                   ordered_class_names: List[str] = ORDERED_CLASS_NAMES):
    
    reverse_label_map = {v: k for k, v in label_map.items()}
    ordered_class_labels = [reverse_label_map[class_name] for class_name in ordered_class_names]

    ordered_class_idxs = torch.tensor([], dtype=torch.int)
    num_instances = target['masks'].shape[0]

    for label in ordered_class_labels:
        ordered_class_idxs = torch.concatenate([ordered_class_idxs, torch.where(target['labels'] == label)[0]])

    # assign the instance IDs in a sequential oder from 1 to num_instances, starting from objects of classes in the provided order 
    # e.g., cage, cell and bead, then take the maximum of instance IDs for overlapping object, since the objects of classes in higher 
    # orders have higher instance IDs, the overlapping pixels are assigned to those object classes
    instance_seg = torch.tensor([i + 1 for i in range(num_instances)]).view(-1, 1, 1).mul(target['masks'][ordered_class_idxs]).max(dim=0)[0]
    
    return instance_seg, {i + 1: target['labels'][ordered_class_idxs[i]].item() for i in range(num_instances)}

# it looks like Mas2Former can support overlapping instances, so there is no need to assign each pixel to only one class
# the function collate_fn_1 blow uses the function generate_instance_segmentation that only assigns each pixel to one class, 
# use collate_fn_2 that is more generic
def collate_fn_1(batch):
    processed_batch = []
    for img_tensor, target in batch:
        instance_seg_tensor, inst2class = generate_instance_segmentation(target)
        inputs = hg_preprocessor([img_tensor], [instance_seg_tensor], instance_id_to_semantic_id=inst2class, return_tensors="pt")
        inputs = {k: v.squeeze() if isinstance(v, torch.Tensor) else v[0] for k,v in inputs.items()}
        processed_batch.append(inputs)
        
    pixel_values = torch.stack([example["pixel_values"] for example in processed_batch])
    pixel_mask = torch.stack([example["pixel_mask"] for example in processed_batch])
    class_labels = [example["class_labels"] for example in processed_batch]
    mask_labels = [example["mask_labels"] for example in processed_batch]
    return {"pixel_values": pixel_values, "pixel_mask": pixel_mask, "class_labels": class_labels, "mask_labels": mask_labels}

def collate_fn_2(batch):
    batch_size = len(batch)
    # each example is a 2-tuple of (normalized image tensor, target dictionary)
    pixel_values = torch.stack([example[0] for example in batch])
    pixel_mask = torch.ones(batch_size, pixel_values.shape[2], pixel_values.shape[3]).to(torch.int)
    class_labels = [example[1]["labels"] for example in batch]
    mask_labels = [example[1]["masks"].to(torch.float32) for example in batch]
    return {"pixel_values": pixel_values, "pixel_mask": pixel_mask, "class_labels": class_labels, "mask_labels": mask_labels}

# define training and validation data loaders
train_data_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size = TRAIN_BATCH_SIZE, shuffle = True, num_workers = 2,
    collate_fn = collate_fn_2)

test_data_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size = 1, shuffle = False, num_workers = 2,
    collate_fn = references.detection.utils.collate_fn)

print('Training data includes %d annotated images.' %len(train_dataset))
print('Test data includes %d annotated images.' %len(test_dataset))

### Visually checking some data

In [ ]:
COLORS = [(0, 0, 0), (0, 0, 255), (255, 0, 0), (0, 255, 0), (255, 255, 0), (255, 0, 255)]

def show_sample(idx, train = True):
    # pick the image from the data set
    if train:
        image, target = train_dataset[idx]
        boxes = target['boxes'].numpy().astype(np.int32)
        labels  = target['labels'].numpy().astype(np.int32)
        masks = target["masks"].numpy().astype(np.uint8)
    else:
        image, target = test_dataset[idx]
        if isinstance(test_dataset, torchvision.datasets.CocoDetection):
            boxes = np.array([t['bbox'] for t in target['annotations']])
            boxes[:, 2] += boxes[:, 0]
            boxes[:, 3] += boxes[:, 1]
            labels = np.array([t['category_id'] for t in target['annotations']])
            masks = np.array([coco_mask_util.decode(t['segmentation']) for t in target['annotations']])
            
        else:
            boxes = target['boxes'].numpy().astype(np.int32)
            labels  = target['labels'].numpy().astype(np.int32)
            masks = target["masks"].numpy().astype(np.uint8)
    # convert the image, bounding  boxes and labels from Tensor to numpy arrays
    # [0, 1] -> 0, 255, np.uint8 format
    image = image.permute(1, 2, 0).mul(torch.tensor(TRANSFORM_STD)).add(torch.tensor(TRANSFORM_MEAN)).mul(255).byte().numpy().copy()
        
    for i in range(len(masks)):
        # the bounding box
        (xtl, ytl, xbr, ybr) = boxes[i]
        # use green color for masks
        color = COLORS[(labels[i] + 1) % len(COLORS)] # add 1 to be consistent with Mask R-CNN colors/labels
        color_mask = color * np.repeat(np.expand_dims(masks[i][ytl:ybr, xtl:xbr], axis=2), 3, axis=2)
        blended = 0.4 * color_mask
        blended[color_mask == 0] = image[ytl:ybr, xtl:xbr][color_mask == 0]
        blended[color_mask > 0] += 0.6 * image[ytl:ybr, xtl:xbr][color_mask > 0]

        # store the blended ROI in the original image
        image[ytl:ybr, xtl:xbr] = blended.astype(np.uint8)
        
        if labels[i] in LABEL_MAP:
            text = LABEL_MAP[labels[i]]
        else:
            print('Incorrect ID was found %s' %labels[i])
            text = 'Unknown'
        
        # add label
        cv2.putText(image, text, (xtl, ytl + 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
        # add the bounding box with yellow color
        color = (255, 255, 0)
        cv2.rectangle(image, (xtl, ytl), (xbr, ybr), color, 1)
        
    print(f"Image size (W, H): {image.shape[1]}, {image.shape[0]}")
    # convert to PIL image to display
    return Image.fromarray(image)

In [ ]:
display(show_sample(900, False))

### Model selection
### From a pretrained model on COCO (scratch)

In [ ]:
model = get_mask2former_instance_segmentation_model_with_dinov2_backbone(
    id2label=LABEL_MAP,
    model_type=DINOV2_BACKBONE_TYPE, 
    with_registers=False
)

# train on the GPU or on the CPU, if a GPU is not available
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

print('Device available:' , device)

# move model to the right device
model.train()
model.to(device)

### Optimizer setting

In [ ]:
# construct an optimizer
params = [p for p in model.parameters() if p.requires_grad]
if OPTIMIZER == 'Adam':
    optimizer = torch.optim.AdamW(params, lr = LEARNING_RATE)
    print('Adam Optimizer is configured for %d epochs' %NUM_EPOCHS)
else:
    optimizer = torch.optim.SGD(params, lr = LEARNING_RATE,
                                momentum = 0.9, weight_decay = 0.0005)
    print('SGD Optimizer is configured for %d epochs' %NUM_EPOCHS)

print('Initial learning rate is set to %s ' %LEARNING_RATE)

# and a learning rate scheduler
if LR_DECAY_STEPS < 1:
    print(f"One-Cyle LR scheduler is configured for {NUM_EPOCHS} epochs with {len(train_data_loader)} steps/epoch")
    lr_scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer=optimizer, 
                                                       max_lr=LEARNING_RATE, 
                                                       epochs=NUM_EPOCHS,
                                                       steps_per_epoch=len(train_data_loader))
else:
    print(f"Step LR scheduler is configured with {LR_DECAY_STEPS} epochs for each step")
    lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=LR_DECAY_STEPS, gamma = 0.1)

In [ ]:
# torch.multiprocessing.set_sharing_strategy('file_system')

In [ ]:
from pycocotools.cocoeval import COCOeval

@torch.inference_mode()
def run_model_and_format_results(model, processor, images, device):

    cpu_device = torch.device("cpu")
    # input image shapes
    org_img_dims = []
    for img in images:
        if isinstance(img, torch.Tensor):
            # image in torch.tensor format of C x H x W
            org_img_dims.append(img.shape[-2:]) 
        elif isinstance(img, np.ndarray):
            # image in numpy array format of H x W x C
            org_img_dims.append(img.shape[:2]) 
        elif isinstance(img, Image.Image):
            org_img_dims.append([img.size[1], img.size[0]]) 
        else:
            print("[ERROR]: Invalid input format of images!")
            return None
    
    # we are passing a list of tensor to the preprocessor instead of PIL images, but it should be fine
    inputs = processor(list(images), return_tensors="pt").to(device)
    outputs = model(**inputs)
    processed_outputs = processor.post_process_instance_segmentation(
        outputs, target_sizes=org_img_dims, return_binary_maps=True
    )

    if len(processed_outputs) == 0:
        # this should not happen and is not expected, return as if the model has not detected anything (for the whole batch)
        return [{'boxes':torch.zeros(0, 4, dtype=torch.float32),
                 'labels': torch.zeros(0, dtype=torch.int64),
                 'scores': torch.zeros(0, dtype=torch.float32),
                 'masks': torch.zeros(0, 1, images[0].shape[-2], images[0].shape[-1], dtype=torch.float32)}] * len(images)
    
    results = []
    
    for sample_index, processed_output in enumerate(processed_outputs):
        
        sample_dict = {}
        instance_to_label_map = {segment['id']: segment['label_id'] for segment in processed_output["segments_info"]}
        instance_to_score_map = {segment['id']: segment['score'] for segment in processed_output["segments_info"]}
        sorted_instance_ids = sorted(instance_to_label_map.keys())
        
        if len(sorted_instance_ids) > 0:

            # processed_output['segmentation'] is of dimension num_detections x H x W
            num_instances = processed_output['segmentation'].shape[0]
            
            if num_instances != len(instance_to_label_map):
                print(f"[WARN]: # of instance masks {num_instances} is not equal to the number of labels {len(instance_to_label_map)}!")
                sorted_instance_ids = [i for i in sorted_instance_ids if i < num_instances]

            # masks should be of dimension num_detections x 1 x H x W
            sample_dict['masks'] = processed_output['segmentation'][sorted_instance_ids].unsqueeze(dim=1).to(cpu_device)
            sample_dict['labels'] =  torch.tensor([instance_to_label_map[i] for i in sorted_instance_ids], dtype=torch.int64)
            sample_dict['scores'] =  torch.tensor([instance_to_score_map[i] for i in sorted_instance_ids], dtype=torch.float)
            boxes = []
            for i in range(sample_dict['masks'].shape[0]):
                pos = torch.where(sample_dict['masks'][i])
                xmin = pos[1].min().item()
                xmax = pos[1].max().item()
                ymin = pos[0].min().item()
                ymax = pos[0].max().item()
                boxes.append([xmin, ymin, xmax, ymax])
            sample_dict['boxes'] =  torch.tensor(boxes, dtype=torch.float)
        else:
            sample_dict = {'boxes':torch.zeros(0, 4, dtype=torch.float32),
                           'labels': torch.zeros(0, dtype=torch.int64),
                           'scores': torch.zeros(0, dtype=torch.float32),
                           'masks': torch.zeros(0, 1, images[sample_index].shape[-2], images[sample_index].shape[-1], dtype=torch.float32)}
            
        results.append(sample_dict)

    return results

@torch.inference_mode()
def evaluate_coco_segm(model, data_loader, processor, device, max_dets=100):
    
    if not isinstance(data_loader.dataset, torchvision.datasets.CocoDetection):
        print(f"[ERROR]: evaluate_coco_segm only supports COCO dataset format (torchvision.datasets.CocoDetection)! \n"
              f"The passed data_loader's dataset type is {type(data_loader.dataset)}. No evaluation is possible")
        return COCOeval(), COCOeval()
        
    n_threads = torch.get_num_threads()
    # FIXME remove this and make paste_masks_in_image run on the GPU
    torch.set_num_threads(1)
    cpu_device = torch.device("cpu")
    
    model.eval()
    model.to(device)
    
    all_results = []
    all_image_ids = []
    model_time = 0

    
    for images, targets in tqdm(data_loader):
        start_time = time.time()
        outputs = run_model_and_format_results(model, processor, images, device)        
        results = {target["image_id"]: output for target, output in zip(targets, outputs)}
        results = convert_preds_to_coco(results)
        all_results.extend(results)
        all_image_ids += [target["image_id"] for target in targets]
        model_time += time.time() - start_time
        
        
    coco_gt = data_loader.dataset.coco   
    coco_dt = coco_gt.loadRes(all_results)  # init predictions api
    
    evaluator_time = time.time()
    
    # bounding box evaluation
    coco_evaluator_bbox = COCOeval(coco_gt, coco_dt, "bbox")
    coco_evaluator_bbox.params.maxDets = [1, 10, max_dets]
    coco_evaluator_bbox.params.imgIds = all_image_ids
    coco_evaluator_bbox.evaluate()
    coco_evaluator_bbox.accumulate()
    coco_evaluator_bbox.summarize()
    
    # segmentation evaluation
    coco_evaluator_segm = COCOeval(coco_gt, coco_dt, "segm")
    coco_evaluator_segm.params.maxDets = [1, 10, max_dets]
    coco_evaluator_segm.params.imgIds = all_image_ids
    coco_evaluator_segm.evaluate()
    coco_evaluator_segm.accumulate()
    coco_evaluator_segm.summarize()
    
    
    evaluator_time = time.time() - evaluator_time
    
    print("model_time:", model_time)
    print("evaluator_time:", evaluator_time)

    torch.set_num_threads(n_threads)
    return coco_evaluator_bbox, coco_evaluator_segm

In [ ]:
def train(model, 
          train_loader, 
          test_loader, 
          preprocessor,
          optimizer, 
          lr_scheduler,
          num_epochs,
          device,
          start_epoch_num=0
         ):
    
    torch.cuda.empty_cache()
    
    # losses and COCO metric
    train_losses: List[float] = []
    aps_0p5_all: List[float] = []
    aps_0p75_all: List[float] = []
    ars_0p5_0p95_all: List[float] = []
   
    # learning rates used for each step (not each epoch as we may use OneCyle scheduling)
    lrs: List[float] = []   
    model = model.to(device)
    
    start_time = time.time()
    
    for epoch in range(start_epoch_num, num_epochs):
        
        since = time.time()
        
        running_loss: float = 0
        
        # training loop
        model.train()
        for i, data in enumerate(tqdm(train_loader)):
            # training phase
            # forward pass 
            outputs = model(
                pixel_values=data["pixel_values"].to(device),
                mask_labels=[masks.to(device) for masks in data["mask_labels"]],
                class_labels=[labels.to(device) for labels in data["class_labels"]],
            )
            
            # backward
            loss = outputs.loss
            loss.backward()
            optimizer.step() # update weight          
            optimizer.zero_grad() # reset gradient
            
            # update the learning rate only after one batch in case of One-Cycle LR scheduler
            lrs.append(lr_scheduler.get_last_lr()[0])
            if isinstance(lr_scheduler, torch.optim.lr_scheduler.OneCycleLR):
                lr_scheduler.step() 
            
            running_loss += loss.item()
        
        # update the learning rate after one full epoch if LR step scheduler is used
        if isinstance(lr_scheduler, torch.optim.lr_scheduler.StepLR):
            lr_scheduler.step() 

        # run COCO evaluation after each training epoch
        # evaluate on the test dataset
        # eval_results = evaluate(model, test_data_loader, device=device, max_dets=5000)
        bbox_metrics, segm_metrics = evaluate_coco_segm(model, test_loader, preprocessor, device=device, max_dets=200)
        # clear CUDA cache
        torch.cuda.empty_cache()
        
        # calculatio mean for each batch
        running_loss /= len(train_loader)                       
         
        print("Epoch:{}/{} ... \n".format(epoch + 1, num_epochs),
              "Train Loss: {:.3f} \n".format(running_loss),
              "Time: {:.2f} m".format((time.time() - since) / 60))

        aps_0p5_all.append(segm_metrics.stats[1])
        aps_0p75_all.append(segm_metrics.stats[2])
        ars_0p5_0p95_all.append(segm_metrics.stats[8])

        # save the COCO Average Precision (AP) for IoU 0.5 and IoU 0.75 so far
        with open(os.path.join(MODEL_PATH, 'coco_results.json'), 'w') as f:
            json.dump([aps_0p5_all, aps_0p75_all, ars_0p5_0p95_all], f)
    
        # saving the checkpoints model after each epoch
        torch.save(model.state_dict(), os.path.join(MODEL_PATH, 'checkpoint_' + str(epoch+1) + '.pt'))
        # save training state to be able to stop and continue a training
        training_state = {'epoch': epoch, 'optimizer': optimizer.state_dict(), 'lr_scheduler': lr_scheduler}
        # save the latest state to be able to continue the training from this point
        torch.save(training_state, os.path.join(MODEL_PATH, 'state.pt'))
        print('Model after epcoh {} has been saved to checkpoint {}, '.format(epoch+1, os.path.join(MODEL_PATH, 'checkpoint_' + str(epoch+1) + '.pt')))
        
    print('Total time: {:.2f} m' .format((time.time()- start_time) / 60))

### Run Training for the specified number of epochs
Specify the epoch number > 0 to start from in case continuing a preveously unfinished training. 

In [ ]:
# starting epoch number if continue a previous training
start_epoch_num = 0
# lists to save the COCO Average Precision (at IoU 0.5 and 0.75 for all object sizes) and Average Recall (at IoU 0.5:0.95 for all sizes)
# they will be loaded from disk if continue from previous training
aps_0p5_all: List[float] = []
aps_0p75_all: List[float] = []
ars_0p5_0p95_all: List[float] = []
if start_epoch_num > 0:
    checkpoint = os.path.join(MODEL_PATH, 'checkpoint_' + str(start_epoch_num) + '.pt')
    state = os.path.join(MODEL_PATH, 'state.pt')
    model_folder_files = os.listdir(MODEL_PATH)
    if ('checkpoint_' + str(start_epoch_num) + '.pt') not in model_folder_files or \
        'state.pt' not in model_folder_files:
        print('Checkpoint and training state could not be found for epoch {}'.format(start_epoch_num-1))
        print('The training will start from epoch 0')
        start_epoch_num = 0
    else:
        # read the state file
        training_state = torch.load(state)
        if training_state['epoch'] != start_epoch_num - 1:
            print('Inconsistent epoch number! In the state dict: {}, specified: {}'.format(training_state['epoch'], start_epoch_num - 1))
            print('The training will start from epoch 0')
            start_epoch_num = 0
        else:
            model.load_state_dict(torch.load(checkpoint))
            torch.optim.Adam.load_state_dict(optimizer, training_state['optimizer'])
            lr_scheduler = training_state['lr_scheduler']
            # load the COCO metrics from previous training
            with open(os.path.join(MODEL_PATH, 'coco_results.json'), 'r') as f:
                aps_0p5_all, aps_0p75_all, ars_0p5_0p95_all = json.load(f)

In [ ]:
hg_preprocessor = Mask2FormerImageProcessor(ignore_index=-1, reduce_labels=False, do_resize=False, do_rescale=False, do_normalize=False)
train(
    model, 
    train_data_loader, 
    test_data_loader, 
    hg_preprocessor,
    optimizer, 
    lr_scheduler,
    NUM_EPOCHS,
    device,
    start_epoch_num
)

### Save the best model with the label map and other parameters

In [ ]:
def get_best_epoch_num(in_aps_0p5_all: List[float], in_aps_0p75_all: List[float], in_ars_0p5_0p95_all: List[float]):
    aps_0p5 = np.array(in_aps_0p5_all)
    aps_0p75 = np.array(in_aps_0p75_all)
    ars = np.array(in_ars_0p5_0p95_all)
    best_ap_idxs = np.argsort(-aps_0p5)
    best_ar_idxs = np.argsort(-ars)
    if best_ap_idxs[0] == best_ar_idxs[0]:
        return best_ap_idxs[0] + 1

    # among the two top APs, pick the one that provides the highest total AP + AR
    idx_1, idx_2 = best_ap_idxs[0], best_ap_idxs[1]
    if aps_0p5[idx_1] + ars[idx_1] > aps_0p5[idx_2] + ars[idx_2]:
        return idx_1 + 1
    return idx_2 + 1

best_epoch_num = get_best_epoch_num(aps_0p5_all, aps_0p75_all, ars_0p5_0p95_all)
print(f"Best epoch number: {best_epoch_num}")
model = get_mask2former_instance_segmentation_model_with_dinov2_backbone(
    id2label=LABEL_MAP,
    model_type=DINOV2_BACKBONE_TYPE, 
    with_registers=False
)
# load the model, the latest saved checkpoint will be loaded
model.load_state_dict(torch.load(os.path.join(MODEL_PATH, 'checkpoint_' + str(best_epoch_num) + '.pt')))
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)


DETECTION_REMAP = None
# DETECTION_REMAP = {"cell-adhered": "cell", "soma": "cell"}


RESIZE = {
    (2000, 1600): (1280, 1024),  # here we keep the (1024, 800) crop size as in Mask R-CNN, the crops will be resizes to (1022, 798)
    (4512, 4512): (2148, 2148),  # here, we are using (1022, 798) crop size with 459 and 348 pixels overlap in x and y
}
# A dictionary with keys as the input (original) image size (width, height) tuple and
# values as the list of coordinates (xtl, ytl, xbr, ybr) of sub-images/crops
# to run YOLOv5 on each
# note that the crop coordinates are with respect to resized image dimensions specified above
CROP_CORNERS = {
    (2000, 1600): [
        [0, 0, 800, 1024],
        [480, 0, 1280, 1024]
    ],
    (4512, 4512): [
        [0, 0, 1022, 798],
        [0, 450, 1022, 1248],
        [0, 900, 1022, 1698],
        [0, 1350, 1022, 2148],
        [563, 0, 1585, 798],
        [563, 450, 1585, 1248],
        [563, 900, 1585, 1698],
        [563, 1350, 1585, 2148],
        [1126, 0, 2148, 798],
        [1126, 450, 2148, 1248],
        [1126, 900, 2148, 1698],
        [1126, 1350, 2148, 2148]
    ]
}
model_param_dict = {}
model_param_dict['model_state_dict'] = model.state_dict()
model_param_dict['label_map'] = LABEL_MAP
model_param_dict['resize_dict'] = RESIZE
model_param_dict['crop_corners_dict'] = CROP_CORNERS
if DETECTION_REMAP is not None:
    model_param_dict['detected_class_names_remap'] = DETECTION_REMAP

torch.save(model_param_dict, os.path.join(MODEL_PATH, 'final.pt'))

### Run Inference
To run inference on a previously trained model, run the cells above up to "Cell size analysis".

In [ ]:
from torchvision.transforms import functional as F

def to_numpy(tensor):
    """
    A function to convert a torch input to numpy array.
    Args:
        tensor (torch tensor).
    Returns:
        Converted to numpy array.
    """
    return tensor.detach().cpu().numpy() if tensor.requires_grad else tensor.cpu().numpy()

def show_predictions(image_pil, predictions, color_depth=12):
    # convert to a numpy array
    image = np.array(image_pil)
    # scale
    image = (255 * image.astype(float) / (2**color_depth - 1)).astype(np.uint8)
    # convert to 3-channels
    image = np.repeat(np.expand_dims(image, axis=2), 3, axis=2)

    boxes = predictions['boxes']
    labels = predictions['labels']
    masks = predictions['masks']

    for i in range(len(masks)):
        # the bounding box
        (xtl, ytl, xbr, ybr) = boxes[i]
        # use green color for masks
        color = COLORS[labels[i] % len(COLORS)]
        mask = masks[i].copy()
        mask[mask >= 0.3] = 1
        mask[mask < 0.3] = 0
        color_mask = color * np.repeat(np.expand_dims(mask, axis=2), 3, axis=2)
        blended = 0.4 * color_mask
        blended[color_mask == 0] = image[ytl:ybr, xtl:xbr][color_mask == 0]
        blended[color_mask > 0] += 0.6 * image[ytl:ybr, xtl:xbr][color_mask > 0]

        # store the blended ROI in the original image
        image[ytl:ybr, xtl:xbr] = blended.astype(np.uint8)
        
        if labels[i] in LABEL_MAP:
            text = LABEL_MAP[labels[i]]
        else:
            print('Incorrect ID was found %s' %labels[i])
            text = 'Unknown'
        
        # add label
        cv2.putText(image, text, (xtl, ytl + 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
        # add the bounding box with yellow color
        color = (255, 255, 0)
        cv2.rectangle(image, (xtl, ytl), (xbr, ybr), color, 1)
        
    # convert to PIL image to display
    return Image.fromarray(image)

# the implementation below is identical to run_model_and_format_results, but we need to call it with a different 
# preprocessor to make sure the image is getting normalized
# we return the results in the same format as Mask R-CNN to be able to reuse the codes written for that model
def predict_batch(model, input_images_list, device):
    
    model.eval()
    model.to(device)

    # convert to 3-channel images if needed, and store the original image dimensions for 
    # post processing
    images_list: List[np.array] = []
    org_img_dims: List[Tuple[int, int]] = []
    
    for img in input_images_list:
        img_shape: tuple = img.shape
        if len(img_shape) < 3:
            images_list.append(cv2.cvtColor(img, cv2.COLOR_GRAY2RGB))
        else:
            images_list.append(img)
        org_img_dims.append(img_shape[:2])
    
    hg_preprocessor = Mask2FormerImageProcessor(ignore_index=0, 
                                                do_resize=True,
                                                size=MODEL_INPUT_SIZE,
                                                size_divisor=14,
                                                reduce_labels=False, 
                                                do_rescale=True,
                                                image_mean=TRANSFORM_MEAN,
                                                image_std=TRANSFORM_STD,
                                                do_normalize=True)
       
    processed_imgs_dict = hg_preprocessor(images_list, return_tensors="pt")
    with torch.no_grad():
        outputs = model(pixel_values=processed_imgs_dict["pixel_values"].to(device))
        processed_outputs = hg_preprocessor.post_process_instance_segmentation(
            outputs, 
            target_sizes=org_img_dims, 
            return_binary_maps=True
        )
   
    if len(processed_outputs) == 0:
        # this should not happen and is not expected, return as if the model has not detected anything (for the whole list of images)
        return [
            {'boxes': [],
             'labels': [],
             'scores': [],
             'masks': []}
        ] * len(images_list)
    
    results = []
    
    for sample_index, processed_output in enumerate(processed_outputs):
        sample_dict = {}
        instance_to_label_map = {segment['id']: segment['label_id'] for segment in processed_output["segments_info"]}
        instance_to_score_map = {segment['id']: segment['score'] for segment in processed_output["segments_info"]}
        sorted_instance_ids = sorted(instance_to_label_map.keys())
        
        if len(sorted_instance_ids) > 0:

            # processed_output['segmentation'] is of dimension num_detections x H x W
            num_instances = processed_output['segmentation'].shape[0]
            
            if num_instances != len(instance_to_label_map):
                print(f"[WARN]: # of instance masks {num_instances} is not equal to the number of labels {len(instance_to_label_map)}!")
                sorted_instance_ids = [i for i in sorted_instance_ids if i < num_instances]

            # masks should be of dimension num_detections x 1 x H x W
            sample_dict['labels'] =  [instance_to_label_map[i] for i in sorted_instance_ids]
            sample_dict['scores'] =  [instance_to_score_map[i] for i in sorted_instance_ids]
            sample_dict['masks'] = to_numpy(processed_output['segmentation'][sorted_instance_ids])
            boxes = []
            masks = [] # masks after restricting them to the size of the bounding box
            for i in range(sample_dict['masks'].shape[0]):
                pos = np.where(sample_dict['masks'][i])
                xtl = pos[1].min()
                xbr = pos[1].max()
                ytl = pos[0].min()
                ybr = pos[0].max()
                boxes.append([xtl, ytl, xbr, ybr])
                masks.append(sample_dict['masks'][i, ytl:ybr, xtl:xbr].astype(float))
            
            sample_dict['boxes'] =  boxes
            sample_dict['masks'] =  masks
            
        else:
            sample_dict = {'boxes': [],
                           'labels': [],
                           'scores': [],
                           'masks': []}
            
        results.append(sample_dict)

    return results

In [ ]:
model = get_mask2former_instance_segmentation_model_with_dinov2_backbone(
    id2label=LABEL_MAP,
    model_type=DINOV2_BACKBONE_TYPE, 
    with_registers=False
)
# load the model, the latest saved checkpoint will be loaded
model.load_state_dict(torch.load(os.path.join(MODEL_PATH, 'checkpoint_' + str(NUM_EPOCHS) + '.pt')))
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)
model.eval()

In [ ]:
# idx = 145123
idx = 1412
# img_path = os.path.join(test_dataset.images_path, test_dataset.imgs[idx])
img_id = test_dataset.coco.getImgIds()[idx]
img_path = os.path.join(test_dataset.root, test_dataset.coco.imgs[img_id]['file_name'])
img = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)

In [ ]:
out = predict_batch(model, [img], device)[0]

In [ ]:
display(show_predictions(img, out, 8))

In [ ]:
display(show_sample(idx, False))

In [ ]:
bbox_metrics, segm_metrics = evaluate_coco_segm(model, test_data_loader, device=device, max_dets=5000)

In [ ]:
test_dataset_org = CellMaskDataset(images_path=TEST_IMAGE_FOLDER, masks_path=TEST_MASK_FOLDER,
                                   annots_in_coco_rle_format = True,
                                   transforms = get_transform(train=False))
test_data_loader_org = torch.utils.data.DataLoader(
        test_dataset_org, batch_size = 1, shuffle = False, num_workers = 2,
        collate_fn = references.detection.utils.collate_fn)
print('Test data includes %d annotated images.' %len(test_dataset_org))
results = evaluate(model, test_data_loader_org, device=device, max_dets=5000)

### Measuring the run-time

In [ ]:
import time
start = time.time()
for i in range(10):
    out = predict_batch(model, [img], device)[0]
print('Running Mask2Former took {} ms'.format((time.time() - start) * 100))